# Transfer Learning com ResNet-18: formigas × abelhas

**Disciplina:** Processamento Digital de Sinais e Imagens (PDSI)

## Contexto

Treinar uma CNN do zero exige **muitas imagens** (o ImageNet tem 1,2 milhão) e muito processamento. Com *transfer learning* (aprendizado por transferência), reaproveitamos uma rede já treinada no ImageNet: as primeiras camadas aprenderam a detectar bordas, texturas e formas, que servem para quase qualquer problema de imagem. Só precisamos **adaptar a última camada** para as nossas classes.

Vamos classificar **formigas e abelhas** com apenas ~120 imagens de treino por classe, comparando três estratégias:

| Estratégia | Pesos iniciais | O que é treinado |
|---|---|---|
| **A. Do zero** | Aleatórios | A rede inteira |
| **B. Extração de características** | ImageNet | Só a nova camada final |
| **C. *Fine-tuning*** | ImageNet | A rede inteira, com taxa de aprendizado pequena |

> 💡 No Colab, ative a GPU em *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.

## Parte 1 – Bibliotecas

In [ ]:
import os, copy, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as TModels
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)                                         # resultados reprodutíveis
device = "cuda" if torch.cuda.is_available() else "cpu"      # usa a GPU, se disponível
print("Dispositivo:", device)

## Parte 2 – Base de dados

A base *hymenoptera* (tutorial oficial do PyTorch) já vem separada em `train` e `val`, com uma pasta por classe. Essa é a organização que o `ImageFolder` espera:

```
hymenoptera_data/
├── train/
│   ├── ants/   (imagens de formigas)
│   └── bees/   (imagens de abelhas)
└── val/
    ├── ants/
    └── bees/
```
Para usar **as suas próprias imagens**, basta montar as pastas nesse formato e trocar o `PASTA_DADOS`.

In [ ]:
if not os.path.isdir("hymenoptera_data"):                    # baixa e descompacta só na primeira execução
    !wget -q https://download.pytorch.org/tutorial/hymenoptera_data.zip
    !unzip -q hymenoptera_data.zip
PASTA_DADOS = "hymenoptera_data"

### Pré-processamento e aumento de dados (*data augmentation*)

No treino, aplicamos recortes e espelhamentos aleatórios: a rede vê variações diferentes da mesma imagem a cada época, o que reduz o *overfitting* com poucas imagens. Na validação, usamos sempre o mesmo recorte central. A normalização usa a média e o desvio padrão do ImageNet, os mesmos do treino original da rede.

In [ ]:
MEDIA, DESVIO = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # estatísticas do ImageNet

transformacoes = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),                    # recorte aleatório redimensionado para 224x224
        transforms.RandomHorizontalFlip(),                    # espelha horizontalmente com 50% de chance
        transforms.ToTensor(),                                # PIL [0, 255] -> tensor [0, 1], formato (C, H, W)
        transforms.Normalize(MEDIA, DESVIO),
    ]),
    "val": transforms.Compose([
        transforms.Resize(256),                               # menor lado com 256 pixels
        transforms.CenterCrop(224),                           # recorte central 224x224
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DESVIO),
    ]),
}

conjuntos = {f: datasets.ImageFolder(os.path.join(PASTA_DADOS, f), transformacoes[f]) for f in ["train", "val"]}
loaders = {f: DataLoader(conjuntos[f], batch_size=16, shuffle=(f == "train"), num_workers=2) for f in ["train", "val"]}
classes = conjuntos["train"].classes                         # nomes das pastas viram os nomes das classes
print("Classes:", classes)
print({f: len(conjuntos[f]) for f in conjuntos})

In [ ]:
def desnormalizar(tensor):
    img = tensor.numpy().transpose(1, 2, 0)                   # (C, H, W) -> (H, W, C) para o Matplotlib
    return np.clip(img * DESVIO + MEDIA, 0, 1)                # desfaz a normalização para exibir

imagens, rotulos = next(iter(loaders["train"]))               # pega um lote de treino
plt.figure(figsize=(12, 3))
for k in range(8):
    plt.subplot(1, 8, k + 1); plt.imshow(desnormalizar(imagens[k])); plt.title(classes[rotulos[k]]); plt.axis("off")
plt.suptitle("Lote de treino (com aumento de dados)"); plt.tight_layout(); plt.show()

## Parte 3 – Função de treinamento

A mesma função será usada nas três estratégias. A cada época ela treina com `train`, avalia com `val` e guarda os pesos da época com a **melhor acurácia de validação**.

In [ ]:
def treinar(modelo, otimizador, epocas=10):
    criterio = nn.CrossEntropyLoss()
    historico = {"perda_treino": [], "acc_treino": [], "perda_val": [], "acc_val": []}
    melhor_acc, melhores_pesos = 0.0, copy.deepcopy(modelo.state_dict())
    inicio = time.time()

    for epoca in range(epocas):
        for fase in ["train", "val"]:
            modelo.train() if fase == "train" else modelo.eval()      # liga/desliga dropout e batch norm
            perda_total, acertos = 0.0, 0
            for x, y in loaders[fase]:
                x, y = x.to(device), y.to(device)
                with torch.set_grad_enabled(fase == "train"):         # gradientes só no treino
                    saidas = modelo(x)
                    perda = criterio(saidas, y)
                    if fase == "train":
                        otimizador.zero_grad(); perda.backward(); otimizador.step()
                perda_total += perda.item() * x.size(0)
                acertos += (saidas.argmax(1) == y).sum().item()
            n = len(conjuntos[fase])
            historico[f"perda_{'treino' if fase == 'train' else 'val'}"].append(perda_total / n)
            historico[f"acc_{'treino' if fase == 'train' else 'val'}"].append(acertos / n)

        acc_val = historico["acc_val"][-1]
        if acc_val > melhor_acc:                                      # guarda a melhor época
            melhor_acc, melhores_pesos = acc_val, copy.deepcopy(modelo.state_dict())
        print(f"Época {epoca + 1:2d}/{epocas} | treino: perda {historico['perda_treino'][-1]:.3f}, "
              f"acc {historico['acc_treino'][-1]:.1%} | val: perda {historico['perda_val'][-1]:.3f}, acc {acc_val:.1%}")

    print(f"Tempo: {time.time() - inicio:.0f} s | Melhor acurácia de validação: {melhor_acc:.1%}")
    modelo.load_state_dict(melhores_pesos)                            # devolve o modelo da melhor época
    return modelo, historico

EPOCAS = 10
historicos = {}

## Parte 4 – Estratégia A: treinar do zero

A ResNet-18 começa com **pesos aleatórios**. Só trocamos a última camada (`fc`), que originalmente tem 1000 saídas (classes do ImageNet), por uma com 2 saídas.

In [ ]:
modelo_zero = TModels.resnet18(weights=None)                  # pesos aleatórios
modelo_zero.fc = nn.Linear(modelo_zero.fc.in_features, len(classes))   # 512 -> 2 classes
modelo_zero = modelo_zero.to(device)

otimizador = torch.optim.SGD(modelo_zero.parameters(), lr=0.01, momentum=0.9)
modelo_zero, historicos["A. Do zero"] = treinar(modelo_zero, otimizador, EPOCAS)

## Parte 5 – Estratégia B: extração de características

Carregamos os **pesos do ImageNet** e **congelamos** todas as camadas (`requires_grad = False`): a rede funciona como um extrator de características fixo. Só a nova camada `fc` é treinada.

In [ ]:
pesos = TModels.ResNet18_Weights.DEFAULT
modelo_extrator = TModels.resnet18(weights=pesos)             # pesos pré-treinados no ImageNet
for parametro in modelo_extrator.parameters():
    parametro.requires_grad = False                          # congela toda a rede
modelo_extrator.fc = nn.Linear(modelo_extrator.fc.in_features, len(classes))  # camada nova: treinável por padrão
modelo_extrator = modelo_extrator.to(device)

treinaveis = sum(p.numel() for p in modelo_extrator.parameters() if p.requires_grad)
total = sum(p.numel() for p in modelo_extrator.parameters())
print(f"Parâmetros treináveis: {treinaveis:,} de {total:,} ({treinaveis / total:.3%})")

otimizador = torch.optim.SGD(modelo_extrator.fc.parameters(), lr=0.01, momentum=0.9)  # otimiza só a fc
modelo_extrator, historicos["B. Extração de características"] = treinar(modelo_extrator, otimizador, EPOCAS)

## Parte 6 – Estratégia C: *fine-tuning*

Também partimos dos pesos do ImageNet, mas agora **toda a rede é ajustada**. Usamos uma taxa de aprendizado **menor** (0.001) para não "apagar" o que a rede já aprendeu.

In [ ]:
modelo_fine = TModels.resnet18(weights=pesos)                 # pesos pré-treinados, todas as camadas treináveis
modelo_fine.fc = nn.Linear(modelo_fine.fc.in_features, len(classes))
modelo_fine = modelo_fine.to(device)

otimizador = torch.optim.SGD(modelo_fine.parameters(), lr=0.001, momentum=0.9)  # taxa 10x menor
modelo_fine, historicos["C. Fine-tuning"] = treinar(modelo_fine, otimizador, EPOCAS)

## Parte 7 – Comparação das estratégias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for nome, h in historicos.items():
    epocas = range(1, len(h["acc_val"]) + 1)
    axes[0].plot(epocas, h["perda_val"], marker="o", label=nome)
    axes[1].plot(epocas, h["acc_val"], marker="o", label=nome)
axes[0].set_title("Perda na validação"); axes[0].set_xlabel("Época")
axes[1].set_title("Acurácia na validação"); axes[1].set_xlabel("Época"); axes[1].set_ylim(0, 1)
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for nome, h in historicos.items():
    print(f"{nome:32s} melhor acurácia de validação: {max(h['acc_val']):.1%}")

### Previsões do melhor modelo na validação

In [ ]:
modelo_fine.eval()
imagens, rotulos = next(iter(loaders["val"]))
with torch.no_grad():
    probs = torch.softmax(modelo_fine(imagens.to(device)), dim=1).cpu()   # probabilidade de cada classe
previstos = probs.argmax(1)

plt.figure(figsize=(14, 6))
for k in range(min(12, len(imagens))):
    cor = "green" if previstos[k] == rotulos[k] else "red"              # verde = acerto, vermelho = erro
    plt.subplot(2, 6, k + 1); plt.imshow(desnormalizar(imagens[k])); plt.axis("off")
    plt.title(f"Previsto: {classes[previstos[k]]} ({probs[k].max():.0%})\nReal: {classes[rotulos[k]]}", color=cor, fontsize=9)
plt.tight_layout(); plt.show()

## Parte 8 – Salvando e reutilizando o modelo

In [ ]:
torch.save(modelo_fine.state_dict(), "resnet18_formigas_abelhas.pth")   # salva apenas os pesos

# Para usar depois: recria a arquitetura com 2 saídas e carrega os pesos salvos
modelo_carregado = TModels.resnet18(weights=None)
modelo_carregado.fc = nn.Linear(512, len(classes))
modelo_carregado.load_state_dict(torch.load("resnet18_formigas_abelhas.pth", map_location=device))
modelo_carregado = modelo_carregado.to(device).eval()
print("Modelo carregado com sucesso!")

## Perguntas para discussão

1. Por que a estratégia A (do zero) teve desempenho tão inferior com a mesma arquitetura e o mesmo número de épocas?
2. Na estratégia B, menos de 0,01% dos parâmetros foram treinados. Por que, mesmo assim, o resultado é bom?
3. Quando vale a pena usar *fine-tuning* em vez de só extração de características? Pense no tamanho da base e na semelhança com o ImageNet (ex.: imagens médicas ou de satélite).
4. Por que o *fine-tuning* usa uma taxa de aprendizado menor?
5. Retire o `RandomResizedCrop` e o `RandomHorizontalFlip` do treino. O que acontece com a diferença entre a acurácia de treino e a de validação?

## Referências

- Tutorial oficial de *transfer learning* do PyTorch: https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
- `ImageFolder` e transformações: https://pytorch.org/vision/stable/datasets.html e https://pytorch.org/vision/stable/transforms.html
- YOSINSKI, J. et al. *How transferable are features in deep neural networks?* In: Advances in Neural Information Processing Systems (NeurIPS), 2014. Disponível em: https://arxiv.org/abs/1411.1792
